# Werkcollege-opdrachten Week 1.2

## Voorbereiding

Importeer in het codeblok hieronder de packages die worden gebruikt om data in te lezen. Geef er ook de gebruikelijke aliassen aan.<br>
N.B.: de 2 reeds geschreven coderegels zorgen ervoor dat eventuele warnings, die code-outputs lelijker maken, uitgezet worden.

In [5]:

import warnings
warnings.simplefilter('ignore')

Zet de volgende bestanden in een makkelijk terug te vinden map:
- go_sales_train.sqlite
- go_crm_train.sqlite
- go_staff_train.sqlite

Bestudeer de bovenste 3 bestanden in DB Browser (SQLite), <a href="https://sqlitebrowser.org/dl/">hier</a> te downloaden. Wat valt je op qua datatypen?<br>

## Databasetabel inlezen

Creëer een databaseconnectie met het bestand go_sales_train.sqlite.

In [1]:
import sqlite3

try:
    sqliteConnection = sqlite3.connect('go_sales_train.sqlite')
    cursor = sqliteConnection.cursor()
    print('Db created and connected succesfull')
except sqlite3.Error as error:
    print('Error occured: ', error)

Db created and connected succesfull


<b>Let goed op:</b><br>
Als je per ongeluk een verkeerde bestandsnaam ingeeft, maakt Python zélf een leeg databasebestand aan! Er ontstaat dan dus een nieuwe .sqlite, en dat is nadrukkelijk <u>niet de bedoeling.</u>

Gebruik de onderstaande sql_query om te achterhalen welke databasetabellen in go_sales_train zitten.

In [2]:
sql_query = "SELECT name FROM sqlite_master WHERE type='table';"

cursor.execute(sql_query)

result = cursor.fetchall()

print(result)
#Vul dit codeblok verder in

[('country',), ('order_details',), ('order_header',), ('order_method',), ('product',), ('product_line',), ('product_type',), ('retailer_site',), ('return_reason',), ('returned_item',), ('sales_branch',), ('sales_staff',)]


Krijg je lege output? Dan heb je per ongeluk een leeg  databasebestand (.sqlite) aangemaakt.<br>
Lees de informatie onder het kopje <u>Let goed op:</u> hierboven nog eens goed door.

Gebruik de gecreëerde databaseconnectie om de resultaten van de volgende query in een DataFrame te zetten:<br>
*SELECT * FROM sales_staff* 

In [18]:
sql_query_2 = "SELECT * FROM sales_staff;"
cursor.execute(sql_query_2)

sales_staff = cursor.fetchall()

print(sales_staff)

[(4, 'Denis', 'Pagé', 'Branch Manager', '+33 1 68 94 52 20', 325, '+33 1 68 94 56 60', 'DPage@grtd123.com', '1996-11-03', 6), (5, 'Élizabeth', 'Michel', 'Level 3 Sales Representative', '+33 1 68 94 52 20', 336, '+33 1 68 94 56 60', 'EMichel@grtd123.com', '1995-06-08', 6), (6, 'Émile', 'Clermont', 'Level 1 Sales Representative', '+33 1 68 94 52 20', 378, '+33 1 68 94 56 60', 'EClermont@grtd123.com', '1998-04-07', 6), (7, 'Étienne', 'Jauvin', 'Level 2 Sales Representative', '+33 1 68 94 52 20', 398, '+33 1 68 94 56 60', 'EJauvin@grtd123.com', '1997-08-16', 6), (12, 'Elsbeth', 'Wiesinger', 'Level 2 Sales Representative', '+(49) 40 663 1990', 1818, '+(49) 40 663 4571', 'EWiesinger@grtd123.com', '1997-02-27', 13), (13, 'Else', 'Mörike', 'Regional Manager', '+(49) 40 663 1990', 1865, '+(49) 40 663 4571', 'EMorike@grtd123.com', '1999-06-02', 13), (14, 'Frank', 'Fuchs', 'Branch Manager', '+(49) 40 663 1990', 1847, '+(49) 40 663 4571', 'FFuchs@grtd123.com', '1997-01-12', 13), (15, 'Gunter', 'Er

## Datumkolommen

Zoals je misschien al hebt gezien in DB Browser, zijn datums als TEXT opgeslagen, en niet als DATE, DATETIME o.i.d. Hier moeten we dus nog even "typische datumkolommen" van maken. Dat doen we met de volgende code:

In [ ]:
import pandas as pd


column_names = [desc[0] for desc in cursor.description]

sales_staff = pd.DataFrame(sales_staff, columns=column_names)

sales_staff['DATE_HIRED'] = pd.to_datetime(sales_staff['DATE_HIRED'])
sales_staff.dtypes

SALES_STAFF_CODE              int64
FIRST_NAME                   object
LAST_NAME                    object
POSITION_EN                  object
WORK_PHONE                   object
EXTENSION                   float64
FAX                          object
EMAIL                        object
DATE_HIRED           datetime64[ns]
SALES_BRANCH_CODE             int64
dtype: object

Als we hier het jaar uit willen halen, schrijven we:

In [25]:
pd.DatetimeIndex(sales_staff['DATE_HIRED']).quarter
# pd.DatetimeIndex(sales_staff['DATE_HIRED']).day
# pd.DatetimeIndex(sales_staff['DATE_HIRED']).year



Index([4, 2, 2, 3, 1, 2, 1, 1, 3, 3,
       ...
       3, 1, 4, 1, 3, 4, 4, 4, 4, 1],
      dtype='int32', name='DATE_HIRED', length=102)

Deze zelfde syntax is te gebruiken voor het extraheren van kwartalen, maanden, weken en dagen. Probeer het maar eens!

## DataFrames uitsplitsen en opbouwen met Series

De volgende 5 kolommen hebben betrekking op de contactdetails van elke medewerker in dit DataFrame:
- SALES_STAFF_CODE
- WORK_PHONE
- EXTENSION
- FAX
- EMAIL

Maak van elk van deze 5 kolommen een serie.

In [45]:
s1 = sales_staff.loc[:, 'SALES_STAFF_CODE']
s2 = sales_staff.loc[:, 'WORK_PHONE']
s3 = sales_staff.loc[:, 'EXTENSION']
s4 = sales_staff.loc[:, 'FAX']
s5 = sales_staff.loc[:, 'EMAIL']

s1



0        4
1        5
2        6
3        7
4       12
      ... 
97     120
98     121
99     122
100    123
101    124
Name: SALES_STAFF_CODE, Length: 102, dtype: int64

Zet allevijf gecreëerde series als kolommen naast elkaar in een DataFrame (*contact_details*). Gebruik pd.concat(...)<br>
Hulpvraag: welke waarde geef je aan de axis-parameter?

In [47]:
contact_details =  pd.concat([s1, s2, s3, s4, s5], axis=1)
contact_details

,SALES_STAFF_CODE,WORK_PHONE,EXTENSION,FAX,EMAIL
0,4,+33 1 68 94 52 20,325.0,+33 1 68 94 56 60,DPage@grtd123.com
1,5,+33 1 68 94 52 20,336.0,+33 1 68 94 56 60,EMichel@grtd123.com
2,6,+33 1 68 94 52 20,378.0,+33 1 68 94 56 60,EClermont@grtd123.com
3,7,+33 1 68 94 52 20,398.0,+33 1 68 94 56 60,EJauvin@grtd123.com
4,12,+(49) 40 663 1990,1818.0,+(49) 40 663 4571,EWiesinger@grtd123.com
...,...,...,...,...,...
97,120,+32 16 20.73.21,1340.0,+32 16 20.73.32,GLaermans@grtd123.com
98,121,+32 16 20.73.21,1642.0,+32 16 20.73.32,FDecree@grtd123.com
99,122,+32 16 20.73.21,1633.0,+32 16 20.73.32,YLattrez@grtd123.com
100,123,+(43) 13 79 56 32,325.0,+(43) 13 79 56 33,WSeefelder@grtd123.com


## Series en DataFrames maken vanuit lists en dictionaries

Met .head(*getal*) kan je de bovenste *getal* rijen van een tabel selecteren.<br>
Selecteer op deze manier de bovenste 5 rijen van *contact_details*.<br>
Sla dit resultaat op in een nieuw DataFrame.

In [50]:
newDF = pd.concat([contact_details.head(5)], axis=1)
newDF

,SALES_STAFF_CODE,WORK_PHONE,EXTENSION,FAX,EMAIL
0,4,+33 1 68 94 52 20,325.0,+33 1 68 94 56 60,DPage@grtd123.com
1,5,+33 1 68 94 52 20,336.0,+33 1 68 94 56 60,EMichel@grtd123.com
2,6,+33 1 68 94 52 20,378.0,+33 1 68 94 56 60,EClermont@grtd123.com
3,7,+33 1 68 94 52 20,398.0,+33 1 68 94 56 60,EJauvin@grtd123.com
4,12,+(49) 40 663 1990,1818.0,+(49) 40 663 4571,EWiesinger@grtd123.com


Aan deze 10 rijen met contactdetails willen we 3 kolommen toevoegen: 'FIRST_LANGUAGE', 'SECOND_LANGUAGE' & 'THIRD_LANGUAGE'.<br>
Iedereens 'First Language' is Engels, afgekort 'EN'. Maak een lijst waarin 5x 'EN' staat.<br>
Converteer deze lijst vervolgens naar een serie en voeg deze horizontaal samen met het resultaat van de vorige opdracht.<br>
Vergeet niet een passende naam te geven aan de nieuw ontstane kolom.

In [54]:
list = ['EN', 'EN', 'EN', 'EN', 'EN']
list = pd.Series(list, name='FIRST_LANGUAGE')

newDF2 = pd.concat([contact_details, list], axis=1)

newDF2

,SALES_STAFF_CODE,WORK_PHONE,EXTENSION,FAX,EMAIL,FIRST_LANGUAGE
0,4,+33 1 68 94 52 20,325.0,+33 1 68 94 56 60,DPage@grtd123.com,EN
1,5,+33 1 68 94 52 20,336.0,+33 1 68 94 56 60,EMichel@grtd123.com,EN
2,6,+33 1 68 94 52 20,378.0,+33 1 68 94 56 60,EClermont@grtd123.com,EN
3,7,+33 1 68 94 52 20,398.0,+33 1 68 94 56 60,EJauvin@grtd123.com,EN
4,12,+(49) 40 663 1990,1818.0,+(49) 40 663 4571,EWiesinger@grtd123.com,EN
...,...,...,...,...,...,...
97,120,+32 16 20.73.21,1340.0,+32 16 20.73.32,GLaermans@grtd123.com,NaN
98,121,+32 16 20.73.21,1642.0,+32 16 20.73.32,FDecree@grtd123.com,NaN
99,122,+32 16 20.73.21,1633.0,+32 16 20.73.32,YLattrez@grtd123.com,NaN
100,123,+(43) 13 79 56 32,325.0,+(43) 13 79 56 33,WSeefelder@grtd123.com,NaN


Maak nu de tweede kolom ('SECOND_LANGUAGE'). Maak daarvoor een dictionary, met daarin...
- Als keys: alle indexen uit het resultaat van het vorige codeblok.
- Als values: bij de eerste 3 elementen 'FR' (Frankrijk), bij de laatste 2 'DE' (Duitsland).

Maak vervolgens ook hier weer een serie van en voeg ook deze weer horizontaal samen met het rsultaat van de vorige opdracht.<br>
Vergeet niet een passende naam te geven aan de nieuw ontstane kolom.

In [64]:
dictionary = {0 : 'FR', 1 : 'FR', 2 : 'FR', 3 :  'DE', 4 : 'DE'}
dictionary = pd.Series(dictionary, name="SECOND_LANGUAGE")

newDF3 = pd.concat([newDF2, dictionary], axis=1)

newDF3

,SALES_STAFF_CODE,WORK_PHONE,EXTENSION,FAX,EMAIL,FIRST_LANGUAGE,SECOND_LANGUAGE
0,4,+33 1 68 94 52 20,325.0,+33 1 68 94 56 60,DPage@grtd123.com,EN,FR
1,5,+33 1 68 94 52 20,336.0,+33 1 68 94 56 60,EMichel@grtd123.com,EN,FR
2,6,+33 1 68 94 52 20,378.0,+33 1 68 94 56 60,EClermont@grtd123.com,EN,FR
3,7,+33 1 68 94 52 20,398.0,+33 1 68 94 56 60,EJauvin@grtd123.com,EN,DE
4,12,+(49) 40 663 1990,1818.0,+(49) 40 663 4571,EWiesinger@grtd123.com,EN,DE
...,...,...,...,...,...,...,...
97,120,+32 16 20.73.21,1340.0,+32 16 20.73.32,GLaermans@grtd123.com,NaN,NaN
98,121,+32 16 20.73.21,1642.0,+32 16 20.73.32,FDecree@grtd123.com,NaN,NaN
99,122,+32 16 20.73.21,1633.0,+32 16 20.73.32,YLattrez@grtd123.com,NaN,NaN
100,123,+(43) 13 79 56 32,325.0,+(43) 13 79 56 33,WSeefelder@grtd123.com,NaN,NaN


Maak ten slotte de derde kolom ('THIRD_LANGUAGE') door een dictionary te maken met daarin...
- Als key: de naam van de nieuwe kolom. Zie je het verschil qua keys met de vorige opdracht?
- Als waarde: een lijst met daarin 'NL', 'NL', 'JPN', 'JPN', 'KOR'.

Converteer deze dictionary nu naar een DataFrame en voeg deze horizontaal samen met het resultaat van de vorige opdracht.<br>
Waarom hoef je hierna de nieuw ontstane kolom geen passende naam meer te geven?

In [68]:
lijst3 = ["NL", "NL", "JPN", "JPN", "KOR"]
dictionary2 = {
    'THIRD_LANGUAGE' : lijst3
}

dictionary2 = pd.DataFrame(dictionary2)

newDF4 = pd.concat([newDF3, dictionary2], axis=1)

newDF4

,SALES_STAFF_CODE,WORK_PHONE,EXTENSION,FAX,EMAIL,FIRST_LANGUAGE,SECOND_LANGUAGE,THIRD_LANGUAGE
0,4,+33 1 68 94 52 20,325.0,+33 1 68 94 56 60,DPage@grtd123.com,EN,FR,NL
1,5,+33 1 68 94 52 20,336.0,+33 1 68 94 56 60,EMichel@grtd123.com,EN,FR,NL
2,6,+33 1 68 94 52 20,378.0,+33 1 68 94 56 60,EClermont@grtd123.com,EN,FR,JPN
3,7,+33 1 68 94 52 20,398.0,+33 1 68 94 56 60,EJauvin@grtd123.com,EN,DE,JPN
4,12,+(49) 40 663 1990,1818.0,+(49) 40 663 4571,EWiesinger@grtd123.com,EN,DE,KOR
...,...,...,...,...,...,...,...,...
97,120,+32 16 20.73.21,1340.0,+32 16 20.73.32,GLaermans@grtd123.com,NaN,NaN,NaN
98,121,+32 16 20.73.21,1642.0,+32 16 20.73.32,FDecree@grtd123.com,NaN,NaN,NaN
99,122,+32 16 20.73.21,1633.0,+32 16 20.73.32,YLattrez@grtd123.com,NaN,NaN,NaN
100,123,+(43) 13 79 56 32,325.0,+(43) 13 79 56 33,WSeefelder@grtd123.com,NaN,NaN,NaN


## Data toevoegen

### Rijen

Gebruik de originele databasetabel *sales_staff*.<br>
Voeg een extra rij toe met eigen invulling. Zorg ervoor dat de index netjes doorloopt.<br>
Hulpvraag: welke waarde geef je nu aan axis?

In [84]:
object1 = {'SALES_STAFF_CODE' : '1301',
           'FIRST_NAME' : 'Bob',
            'LAST_NAME' :  'de Bouwer',
             'POSITION_EN' : 'Verkoper',
              'WORK_PHONE' :  '+31 1 45 67 89 01'}
object1 = pd.DataFrame([object1])
newDF5 = pd.concat([sales_staff, object1], axis=0, ignore_index=True)

newDF5

,SALES_STAFF_CODE,FIRST_NAME,LAST_NAME,POSITION_EN,WORK_PHONE,EXTENSION,FAX,EMAIL,DATE_HIRED,SALES_BRANCH_CODE
0,4,Denis,Pagé,Branch Manager,+33 1 68 94 52 20,325.0,+33 1 68 94 56 60,DPage@grtd123.com,1996-11-03,6.0
1,5,Élizabeth,Michel,Level 3 Sales Representative,+33 1 68 94 52 20,336.0,+33 1 68 94 56 60,EMichel@grtd123.com,1995-06-08,6.0
2,6,Émile,Clermont,Level 1 Sales Representative,+33 1 68 94 52 20,378.0,+33 1 68 94 56 60,EClermont@grtd123.com,1998-04-07,6.0
3,7,Étienne,Jauvin,Level 2 Sales Representative,+33 1 68 94 52 20,398.0,+33 1 68 94 56 60,EJauvin@grtd123.com,1997-08-16,6.0
4,12,Elsbeth,Wiesinger,Level 2 Sales Representative,+(49) 40 663 1990,1818.0,+(49) 40 663 4571,EWiesinger@grtd123.com,1997-02-27,13.0
...,...,...,...,...,...,...,...,...,...,...
98,121,François,De Crée,Level 1 Sales Representative,+32 16 20.73.21,1642.0,+32 16 20.73.32,FDecree@grtd123.com,1999-12-01,38.0
99,122,Yvette,Lattrez,Level 3 Sales Representative,+32 16 20.73.21,1633.0,+32 16 20.73.32,YLattrez@grtd123.com,1999-12-09,38.0
100,123,Willi,Seefelder,Level 2 Sales Representative,+(43) 13 79 56 32,325.0,+(43) 13 79 56 33,WSeefelder@grtd123.com,1998-10-28,39.0
101,124,Sabine,Grüner,Level 3 Sales Representative,+(43) 13 79 56 32,348.0,+(43) 13 79 56 33,SGruner@grtd123.com,1998-02-18,39.0


### Kolommen

Voeg een kolom *FULL_NAME* toe die de waarden van *FIRST_NAME* en *LAST_NAME* achter elkaar zet, gescheiden door een spatie.

In [86]:
newDF5['FULL_NAME'] = newDF5['FIRST_NAME'] + " " + newDF5['LAST_NAME']

newDF5

,SALES_STAFF_CODE,FIRST_NAME,LAST_NAME,POSITION_EN,WORK_PHONE,EXTENSION,FAX,EMAIL,DATE_HIRED,SALES_BRANCH_CODE,FULL_NAME
0,4,Denis,Pagé,Branch Manager,+33 1 68 94 52 20,325.0,+33 1 68 94 56 60,DPage@grtd123.com,1996-11-03,6.0,Denis Pagé
1,5,Élizabeth,Michel,Level 3 Sales Representative,+33 1 68 94 52 20,336.0,+33 1 68 94 56 60,EMichel@grtd123.com,1995-06-08,6.0,Élizabeth Michel
2,6,Émile,Clermont,Level 1 Sales Representative,+33 1 68 94 52 20,378.0,+33 1 68 94 56 60,EClermont@grtd123.com,1998-04-07,6.0,Émile Clermont
3,7,Étienne,Jauvin,Level 2 Sales Representative,+33 1 68 94 52 20,398.0,+33 1 68 94 56 60,EJauvin@grtd123.com,1997-08-16,6.0,Étienne Jauvin
4,12,Elsbeth,Wiesinger,Level 2 Sales Representative,+(49) 40 663 1990,1818.0,+(49) 40 663 4571,EWiesinger@grtd123.com,1997-02-27,13.0,Elsbeth Wiesinger
...,...,...,...,...,...,...,...,...,...,...,...
98,121,François,De Crée,Level 1 Sales Representative,+32 16 20.73.21,1642.0,+32 16 20.73.32,FDecree@grtd123.com,1999-12-01,38.0,François De Crée
99,122,Yvette,Lattrez,Level 3 Sales Representative,+32 16 20.73.21,1633.0,+32 16 20.73.32,YLattrez@grtd123.com,1999-12-09,38.0,Yvette Lattrez
100,123,Willi,Seefelder,Level 2 Sales Representative,+(43) 13 79 56 32,325.0,+(43) 13 79 56 33,WSeefelder@grtd123.com,1998-10-28,39.0,Willi Seefelder
101,124,Sabine,Grüner,Level 3 Sales Representative,+(43) 13 79 56 32,348.0,+(43) 13 79 56 33,SGruner@grtd123.com,1998-02-18,39.0,Sabine Grüner


## Data wijzigen

### Datatypen

Door het attribuut .dtypes van een DataFrame op te vragen krijg je een serie die per kolom het datatype weergeeft. Doe dit bij de originele databasetabel *sales_staff*

In [87]:
sales_staff.dtypes

SALES_STAFF_CODE              int64
FIRST_NAME                   object
LAST_NAME                    object
POSITION_EN                  object
WORK_PHONE                   object
EXTENSION                   float64
FAX                          object
EMAIL                        object
DATE_HIRED           datetime64[ns]
SALES_BRANCH_CODE             int64
dtype: object

Hier valt op dat elke kolom het datatype 'object' heeft: Python leest dus alles als string. Wiskundige operaties zijn hierdoor niet mogelijk.<br>
We kunnen proberen om kolommen met getallen, bijvoorbeeld de 'extension', te converteren naar een int. Zie onderstaande code.

In [3]:
sales_staff['EXTENSION'] = sales_staff['EXTENSION'].astype(int)
sales_staff.dtypes

NameError: name 'sales_staff' is not defined

Dit lukt echter niet, omdat er in de kolom 'EXTENSION' lege waarden zitten die niet geconverteerd kunnen worden naar een int.<br>
Wél kan je deze naar een float converteren, zie onderstaande code:

In [88]:
sales_staff['EXTENSION'] = sales_staff['EXTENSION'].astype(float)
sales_staff.dtypes

SALES_STAFF_CODE              int64
FIRST_NAME                   object
LAST_NAME                    object
POSITION_EN                  object
WORK_PHONE                   object
EXTENSION                   float64
FAX                          object
EMAIL                        object
DATE_HIRED           datetime64[ns]
SALES_BRANCH_CODE             int64
dtype: object

Als we in de rij van 'EXTENSION' kijken zien we dat de conversie van het datatype nu is gelukt.<br>
Dit is <b>randvoorwaardelijk</b> voor het uitvoeren van wiskundige operaties.<br>

### Rijen

Zorg er nu voor dat bij alle extensions 1 wordt opgeteld.

In [92]:
sales_staff['EXTENSION'] = sales_staff['EXTENSION'] +1
sales_staff

,SALES_STAFF_CODE,FIRST_NAME,LAST_NAME,POSITION_EN,WORK_PHONE,EXTENSION,FAX,EMAIL,DATE_HIRED,SALES_BRANCH_CODE
0,4,Denis,Pagé,Branch Manager,+33 1 68 94 52 20,328.0,+33 1 68 94 56 60,DPage@grtd123.com,1996-11-03,6
1,5,Élizabeth,Michel,Level 3 Sales Representative,+33 1 68 94 52 20,339.0,+33 1 68 94 56 60,EMichel@grtd123.com,1995-06-08,6
2,6,Émile,Clermont,Level 1 Sales Representative,+33 1 68 94 52 20,381.0,+33 1 68 94 56 60,EClermont@grtd123.com,1998-04-07,6
3,7,Étienne,Jauvin,Level 2 Sales Representative,+33 1 68 94 52 20,401.0,+33 1 68 94 56 60,EJauvin@grtd123.com,1997-08-16,6
4,12,Elsbeth,Wiesinger,Level 2 Sales Representative,+(49) 40 663 1990,1821.0,+(49) 40 663 4571,EWiesinger@grtd123.com,1997-02-27,13
...,...,...,...,...,...,...,...,...,...,...
97,120,Giele,Laermans,Level 1 Sales Representative,+32 16 20.73.21,1343.0,+32 16 20.73.32,GLaermans@grtd123.com,1999-11-15,38
98,121,François,De Crée,Level 1 Sales Representative,+32 16 20.73.21,1645.0,+32 16 20.73.32,FDecree@grtd123.com,1999-12-01,38
99,122,Yvette,Lattrez,Level 3 Sales Representative,+32 16 20.73.21,1636.0,+32 16 20.73.32,YLattrez@grtd123.com,1999-12-09,38
100,123,Willi,Seefelder,Level 2 Sales Representative,+(43) 13 79 56 32,328.0,+(43) 13 79 56 33,WSeefelder@grtd123.com,1998-10-28,39


Elke 'Branch Manager' wordt nu 'General Manager'. Schrijf code zodat deze wijziging doorgevoerd wordt in het DataFrame.

In [ ]:
sales_staff['POSITION_EN'].where(sales_staff['POSITION_EN'] == 'Branch Manager').replace('Branch Manager', 'General Manager')

0      General Manager
1                  NaN
2                  NaN
3                  NaN
4                  NaN
            ...       
97                 NaN
98                 NaN
99                 NaN
100                NaN
101                NaN
Name: POSITION_EN, Length: 102, dtype: object

### Kolommen

Verander de kolomnaam 'POSITION_EN' naar 'POSITION'.

In [105]:
sales_staff.rename({'POSITION_EN' : 'POSITION'}, axis='columns')

,SALES_STAFF_CODE,FIRST_NAME,LAST_NAME,POSITION,WORK_PHONE,EXTENSION,FAX,EMAIL,DATE_HIRED,SALES_BRANCH_CODE
0,4,Denis,Pagé,Branch Manager,+33 1 68 94 52 20,328.0,+33 1 68 94 56 60,DPage@grtd123.com,1996-11-03,6
1,5,Élizabeth,Michel,Level 3 Sales Representative,+33 1 68 94 52 20,339.0,+33 1 68 94 56 60,EMichel@grtd123.com,1995-06-08,6
2,6,Émile,Clermont,Level 1 Sales Representative,+33 1 68 94 52 20,381.0,+33 1 68 94 56 60,EClermont@grtd123.com,1998-04-07,6
3,7,Étienne,Jauvin,Level 2 Sales Representative,+33 1 68 94 52 20,401.0,+33 1 68 94 56 60,EJauvin@grtd123.com,1997-08-16,6
4,12,Elsbeth,Wiesinger,Level 2 Sales Representative,+(49) 40 663 1990,1821.0,+(49) 40 663 4571,EWiesinger@grtd123.com,1997-02-27,13
...,...,...,...,...,...,...,...,...,...,...
97,120,Giele,Laermans,Level 1 Sales Representative,+32 16 20.73.21,1343.0,+32 16 20.73.32,GLaermans@grtd123.com,1999-11-15,38
98,121,François,De Crée,Level 1 Sales Representative,+32 16 20.73.21,1645.0,+32 16 20.73.32,FDecree@grtd123.com,1999-12-01,38
99,122,Yvette,Lattrez,Level 3 Sales Representative,+32 16 20.73.21,1636.0,+32 16 20.73.32,YLattrez@grtd123.com,1999-12-09,38
100,123,Willi,Seefelder,Level 2 Sales Representative,+(43) 13 79 56 32,328.0,+(43) 13 79 56 33,WSeefelder@grtd123.com,1998-10-28,39


## Data verwijderen

### Rijen

De medewerkers op indexen 99, 100 en 101 hebben helaas ontslag genomen.<br>
Verwijder de bijbehorende rijen uit het DataFrame. Gebruik slechts één keer de .drop()-methode.

In [111]:
sales_staff.drop([99, 100, 101])

,SALES_STAFF_CODE,FIRST_NAME,LAST_NAME,POSITION_EN,WORK_PHONE,EXTENSION,FAX,EMAIL,DATE_HIRED,SALES_BRANCH_CODE
0,4,Denis,Pagé,Branch Manager,+33 1 68 94 52 20,328.0,+33 1 68 94 56 60,DPage@grtd123.com,1996-11-03,6
1,5,Élizabeth,Michel,Level 3 Sales Representative,+33 1 68 94 52 20,339.0,+33 1 68 94 56 60,EMichel@grtd123.com,1995-06-08,6
2,6,Émile,Clermont,Level 1 Sales Representative,+33 1 68 94 52 20,381.0,+33 1 68 94 56 60,EClermont@grtd123.com,1998-04-07,6
3,7,Étienne,Jauvin,Level 2 Sales Representative,+33 1 68 94 52 20,401.0,+33 1 68 94 56 60,EJauvin@grtd123.com,1997-08-16,6
4,12,Elsbeth,Wiesinger,Level 2 Sales Representative,+(49) 40 663 1990,1821.0,+(49) 40 663 4571,EWiesinger@grtd123.com,1997-02-27,13
...,...,...,...,...,...,...,...,...,...,...
94,117,Frank,Jever,Level 3 Sales Representative,+(41) 17 12 13 14,1157.0,+(41) 17 12 13 20,FJever@grtd123.com,1999-10-18,37
95,118,Gianni,Vertemati,Level 1 Sales Representative,+(41) 17 12 13 14,1257.0,+(41) 17 12 13 20,GVertemati@grtd123.com,2000-02-10,37
96,119,Gracy,Gellens,Branch Manager,+32 16 20.73.21,1352.0,+32 16 20.73.32,GGellens@grtd123.com,1999-09-10,38
97,120,Giele,Laermans,Level 1 Sales Representative,+32 16 20.73.21,1343.0,+32 16 20.73.32,GLaermans@grtd123.com,1999-11-15,38


### Kolommen

Faxen zijn inmiddels ouderwets: niemand gebruikt zijn/haar faxnummer nog.<br>
Verwijder de bijbehorende kolom uit het DataFrame.

In [112]:
sales_staff.drop(columns="FAX")

,SALES_STAFF_CODE,FIRST_NAME,LAST_NAME,POSITION_EN,WORK_PHONE,EXTENSION,EMAIL,DATE_HIRED,SALES_BRANCH_CODE
0,4,Denis,Pagé,Branch Manager,+33 1 68 94 52 20,328.0,DPage@grtd123.com,1996-11-03,6
1,5,Élizabeth,Michel,Level 3 Sales Representative,+33 1 68 94 52 20,339.0,EMichel@grtd123.com,1995-06-08,6
2,6,Émile,Clermont,Level 1 Sales Representative,+33 1 68 94 52 20,381.0,EClermont@grtd123.com,1998-04-07,6
3,7,Étienne,Jauvin,Level 2 Sales Representative,+33 1 68 94 52 20,401.0,EJauvin@grtd123.com,1997-08-16,6
4,12,Elsbeth,Wiesinger,Level 2 Sales Representative,+(49) 40 663 1990,1821.0,EWiesinger@grtd123.com,1997-02-27,13
...,...,...,...,...,...,...,...,...,...
97,120,Giele,Laermans,Level 1 Sales Representative,+32 16 20.73.21,1343.0,GLaermans@grtd123.com,1999-11-15,38
98,121,François,De Crée,Level 1 Sales Representative,+32 16 20.73.21,1645.0,FDecree@grtd123.com,1999-12-01,38
99,122,Yvette,Lattrez,Level 3 Sales Representative,+32 16 20.73.21,1636.0,YLattrez@grtd123.com,1999-12-09,38
100,123,Willi,Seefelder,Level 2 Sales Representative,+(43) 13 79 56 32,328.0,WSeefelder@grtd123.com,1998-10-28,39
